# Stochastic Optimization — Exercises

8 exercises covering SGD convergence, mini-batch variance, SVRG, SAGA, distributed SGD, and the generalization gap.

| Format            | Description                                  |
| ----------------- | -------------------------------------------- |
| **Problem**       | Markdown cell with task description          |
| **Your Solution** | Code cell with scaffolding                   |
| **Solution**      | Code cell with reference solution and checks |

### Difficulty Levels

| Level | Exercises | Focus                |
| ----- | --------- | -------------------- |
| ★     | 1-3       | Core mechanics       |
| ★★    | 4-6       | Deeper theory        |
| ★★★   | 7-8       | AI / ML applications |

### Topic Map

| Topic    | Exercise |
| -------- | -------- |
| SGD unbiasedness | 1 |
| SGD convergence rate | 2 |
| Mini-batch variance | 3 |
| SVRG analysis | 4 |
| Linear scaling rule | 5 |
| Momentum in stochastic setting | 6 |
| Distributed SGD | 7 |
| SGD implicit bias | 8 |

In [ ]:
import numpy as np
import numpy.linalg as la

try:
    import matplotlib.pyplot as plt
    HAS_MPL = True
except ImportError:
    HAS_MPL = False

COLORS = {
    'primary': '#0077BB',
    'secondary': '#EE7733',
    'tertiary': '#009988',
    'error': '#CC3311',
    'neutral': '#555555',
    'highlight': '#EE3377',
}

np.set_printoptions(precision=6, suppress=True)
np.random.seed(42)


def header(title):
    print('\n' + '=' * len(title))
    print(title)
    print('=' * len(title))


def check_close(name, got, expected, tol=1e-8):
    ok = np.allclose(got, expected, atol=tol, rtol=tol)
    print(f"{'PASS' if ok else 'FAIL'} — {name}")
    if not ok:
        print('  expected:', expected)
        print('  got     :', got)
    return ok


def check_true(name, cond):
    print(f"{'PASS' if cond else 'FAIL'} — {name}")
    return cond


def sgd_step(x, grad_fn, lr):
    return x - lr * grad_fn(x)


print('Setup complete.')


---

### Exercise 1: SGD Unbiasedness ★

Prove that the mini-batch gradient estimator is unbiased and compute its variance.

**(a)** Show that $\mathbb{E}[\mathbf{g}_B(\boldsymbol{\theta})] = \nabla F(\boldsymbol{\theta})$ where $\mathbf{g}_B = \frac{1}{B}\sum_{j=1}^B \nabla f_{i_j}(\boldsymbol{\theta})$.

**(b)** Show that $\text{Var}(\mathbf{g}_B) = \sigma^2/B$.

**(c)** Verify numerically on a quadratic problem.

In [ ]:
# Exercise 1: Your Solution
np.random.seed(42)
n, d = 1000, 10
A = np.random.randn(n, d)
x = np.random.randn(d)
b = A @ x + 0.1 * np.random.randn(n)

def f_i(x, i): return 0.5 * (A[i] @ x - b[i])**2
def grad_f_i(x, i): return (A[i] @ x - b[i]) * A[i]

full_grad = np.mean([grad_f_i(x, i) for i in range(n)], axis=0)

# Estimate E[g_B] and Var(g_B) for different B
batch_sizes = [1, 4, 16, 64]
for B in batch_sizes:
    grads = []
    for _ in range(1000):
        idx = np.random.randint(0, n, size=B)
        g = np.mean([grad_f_i(x, i) for i in idx], axis=0)
        grads.append(g)
    grads = np.array(grads)
    mean_g = np.mean(grads, axis=0)
    var_g = np.mean(np.sum((grads - full_grad)**2, axis=1))
    print(f'B={B:3d}: E[g]={mean_g[0]:.4f} (true={full_grad[0]:.4f}), Var={var_g:.6f}')

In [ ]:
# Exercise 1: Solution
np.random.seed(42)
n, d = 1000, 10
A = np.random.randn(n, d)
x = np.random.randn(d)
b = A @ x + 0.1 * np.random.randn(n)

def grad_f_i(x, i): return (A[i] @ x - b[i]) * A[i]

full_grad = np.mean([grad_f_i(x, i) for i in range(n)], axis=0)

header('Exercise 1: SGD Unbiasedness')
batch_sizes = [1, 4, 16, 64]
variances = []
for B in batch_sizes:
    grads = []
    for _ in range(1000):
        idx = np.random.randint(0, n, size=B)
        g = np.mean([grad_f_i(x, i) for i in idx], axis=0)
        grads.append(g)
    grads = np.array(grads)
    mean_g = np.mean(grads, axis=0)
    var_g = np.mean(np.sum((grads - full_grad)**2, axis=1))
    variances.append(var_g)
    print(f'B={B:3d}: E[g]≈{mean_g[0]:.4f} (true={full_grad[0]:.4f}), Var={var_g:.6f}')

variances = np.array(variances)
check_close('unbiased: E[g] ≈ full_grad', np.mean([np.mean(np.array([np.mean([grad_f_i(x, i) for i in np.random.randint(0, n, size=B)], axis=0) for _ in range(100)])) for B in [1, 4]], axis=0), full_grad, tol=0.1)
check_true('variance decreases with B', variances[-1] < variances[0])

# Verify σ²/B scaling
ratio = variances[0] / variances[-1]
expected_ratio = batch_sizes[-1] / batch_sizes[0]
print(f'\nVariance ratio B=64/B=1: {ratio:.1f}x (expected {expected_ratio}x)')
check_close('variance scales as 1/B', ratio, expected_ratio, tol=expected_ratio * 0.3)

print('\nTakeaway: Mini-batch gradient is unbiased with variance σ²/B.')

---

### Exercise 2: SGD Convergence Rate ★

For $f(x) = \frac{1}{2}x^2$ with stochastic gradient $g_t = x_t + \xi_t$ where $\xi_t \sim \mathcal{N}(0, \sigma^2)$, analyze convergence with $\eta_t = \frac{1}{t+1}$.

**(a)** Show that $\mathbb{E}[x_t^2] = O(1/t)$.

**(b)** Verify numerically.

In [ ]:
# Exercise 2: Your Solution
sigma = 1.0
T = 10000
n_runs = 100

err_all = np.zeros(T)
for run in range(n_runs):
    x = 1.0
    for t in range(T):
        eta = 1.0 / (t + 1)
        g = x + sigma * np.random.randn()
        x = x - eta * g
        err_all[t] += x**2

err_all /= n_runs

print(f'E[x_T²] = {err_all[-1]:.6f}')
print(f'1/T = {1/T:.6f}')
print(f'T * E[x_T²] = {T * err_all[-1]:.4f} (should be O(1))')

In [ ]:
# Exercise 2: Solution
sigma = 1.0
T = 10000
n_runs = 100

err_all = np.zeros(T)
for run in range(n_runs):
    x = 1.0
    for t in range(T):
        eta = 1.0 / (t + 1)
        g = x + sigma * np.random.randn()
        x = x - eta * g
        err_all[t] += x**2

err_all /= n_runs

header('Exercise 2: SGD Convergence Rate')
print(f'E[x_T²] = {err_all[-1]:.6f}')
print(f'1/T = {1/T:.6f}')
print(f'T * E[x_T²] = {T * err_all[-1]:.4f}')

# Check O(1/t) rate
T_vals = np.array([100, 500, 1000, 5000, 10000])
print(f"\n{'T':>6} | {'E[x²]':>12} | {'T·E[x²]':>10}")
print('-' * 35)
for T_val in T_vals:
    print(f'{T_val:>6} | {err_all[T_val-1]:>12.6f} | {T_val*err_all[T_val-1]:>10.4f}')

check_true('T * E[x²] is bounded', T * err_all[-1] < 10)
check_true('error decreases', err_all[-1] < err_all[0])

print('\nTakeaway: SGD with η_t = 1/(t+1) converges at rate O(1/t) for strongly convex problems.')

---

### Exercise 3: Mini-Batch Variance ★

For a dataset with $n = 10,000$ and gradient variance $\sigma^2 = 4$, compute the variance for different batch sizes.

**(a)** Compute the variance for $B \in \{1, 32, 128, 1024\}$.

**(b)** Plot variance vs. batch size on a log-log scale.

**(c)** Verify the $1/B$ scaling.

In [ ]:
# Exercise 3: Your Solution
np.random.seed(42)
n, d = 10000, 20
A = np.random.randn(n, d)
x = np.random.randn(d)
b = A @ x + np.random.randn(n)

full_grad = A.T @ (A @ x - b) / n

batch_sizes = [1, 32, 128, 1024]
variances = []

for B in batch_sizes:
    grads = []
    for _ in range(200):
        idx = np.random.randint(0, n, size=B)
        g = A[idx].T @ (A[idx] @ x - b[idx]) / B
        grads.append(g)
    grads = np.array(grads)
    var = np.mean(np.sum((grads - full_grad)**2, axis=1))
    variances.append(var)

print(f"{'B':>6} | {'Variance':>12} | {'σ²/Var':>10}")
print('-' * 35)
for B, v in zip(batch_sizes, variances):
    print(f'{B:>6} | {v:>12.6f} | {variances[0]/v:>10.1f}x')

In [ ]:
# Exercise 3: Solution
np.random.seed(42)
n, d = 10000, 20
A = np.random.randn(n, d)
x = np.random.randn(d)
b = A @ x + np.random.randn(n)

full_grad = A.T @ (A @ x - b) / n

batch_sizes = [1, 32, 128, 1024]
variances = []

for B in batch_sizes:
    grads = []
    for _ in range(200):
        idx = np.random.randint(0, n, size=B)
        g = A[idx].T @ (A[idx] @ x - b[idx]) / B
        grads.append(g)
    grads = np.array(grads)
    var = np.mean(np.sum((grads - full_grad)**2, axis=1))
    variances.append(var)

header('Exercise 3: Mini-Batch Variance')
variances = np.array(variances)

print(f"{'B':>6} | {'Variance':>12} | {'σ²/Var':>10}")
print('-' * 35)
for B, v in zip(batch_sizes, variances):
    print(f'{B:>6} | {v:>12.6f} | {variances[0]/v:>10.1f}x')

# Verify 1/B scaling
expected_ratios = np.array(batch_sizes) / batch_sizes[0]
actual_ratios = variances[0] / variances
check_close('variance scales as 1/B', actual_ratios, expected_ratios, tol=expected_ratios * 0.3)

if HAS_MPL:
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(batch_sizes, variances, 'o-', color=COLORS['primary'])
    ax.plot(batch_sizes, variances[0] / np.array(batch_sizes), '--', color=COLORS['neutral'], label='σ²/B')
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_title('Gradient variance vs batch size')
    ax.set_xlabel('Batch size')
    ax.set_ylabel('Variance')
    ax.legend()
    fig.tight_layout()
    plt.show()

print('\nTakeaway: Mini-batch variance decreases as σ²/B.')

---

### Exercise 4: SVRG Analysis ★★

Implement SVRG for logistic regression and verify linear convergence.

**(a)** Implement the SVRG algorithm.

**(b)** Compare convergence rate with vanilla SGD.

**(c)** Verify that SVRG converges linearly.

In [ ]:
# Exercise 4: Your Solution
np.random.seed(42)
n, d = 500, 20
X = np.random.randn(n, d)
w_true = np.random.randn(d)
y = (X @ w_true > 0).astype(float) * 2 - 1

def sigmoid(z): return 1.0 / (1.0 + np.exp(-np.clip(z, -500, 500)))

def f_i(w, i):
    z = y[i] * X[i] @ w
    return np.log(1 + np.exp(-np.clip(z, -500, 500)))

def grad_f_i(w, i):
    z = y[i] * X[i] @ w
    s = sigmoid(-z)
    return -y[i] * s * X[i]

def full_grad(w):
    return np.mean([grad_f_i(w, i) for i in range(n)], axis=0)

def full_loss(w):
    return np.mean([f_i(w, i) for i in range(n)])

# SVRG
w = np.zeros(d)
m = 2 * n
eta = 0.01
n_epochs = 15
loss_hist = []

for s in range(n_epochs):
    full_g = full_grad(w)
    w_snapshot = w.copy()
    for t in range(m):
        i = np.random.randint(n)
        g_current = grad_f_i(w, i)
        g_snapshot = grad_f_i(w_snapshot, i)
        g_svrg = g_current - g_snapshot + full_g
        w = w - eta * g_svrg
    loss_hist.append(full_loss(w))

loss_hist = np.array(loss_hist)
print(f'SVRG: Initial={loss_hist[0]:.6f}, Final={loss_hist[-1]:.6f}')
for s in range(1, len(loss_hist)):
    ratio = loss_hist[s] / loss_hist[s-1] if loss_hist[s-1] > 0 else 0
    print(f'  Epoch {s}: ratio={ratio:.4f}')

In [ ]:
# Exercise 4: Solution
np.random.seed(42)
n, d = 500, 20
X = np.random.randn(n, d)
w_true = np.random.randn(d)
y = (X @ w_true > 0).astype(float) * 2 - 1


def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-np.clip(z, -500, 500)))


def f_i(w, i):
    z = y[i] * X[i] @ w
    return np.log(1 + np.exp(-np.clip(z, -500, 500)))


def grad_f_i(w, i):
    z = y[i] * X[i] @ w
    s = sigmoid(-z)
    return -y[i] * s * X[i]


def full_grad(w):
    return np.mean([grad_f_i(w, i) for i in range(n)], axis=0)


def full_loss(w):
    return np.mean([f_i(w, i) for i in range(n)])


w = np.zeros(d)
m = 2 * n
eta = 0.02
n_epochs = 15
loss_hist = [full_loss(w)]

for s in range(n_epochs):
    full_g = full_grad(w)
    w_snapshot = w.copy()
    for t in range(m):
        i = np.random.randint(n)
        g_current = grad_f_i(w, i)
        g_snapshot = grad_f_i(w_snapshot, i)
        g_svrg = g_current - g_snapshot + full_g
        w = w - eta * g_svrg
    loss_hist.append(full_loss(w))

loss_hist = np.array(loss_hist)

header('Exercise 4: SVRG Analysis')
print(f'SVRG: Initial={loss_hist[0]:.6f}, Final={loss_hist[-1]:.6f}')

ratios = []
for s in range(1, len(loss_hist)):
    ratio = loss_hist[s] / loss_hist[s-1] if loss_hist[s-1] > 0 else 0.0
    ratios.append(ratio)
    print(f'  Epoch {s}: loss={loss_hist[s]:.6f}, ratio={ratio:.4f}')

check_true('loss decreases over epochs', np.all(np.diff(loss_hist) <= 1e-8))
check_true('significant loss reduction', loss_hist[-1] < 0.5 * loss_hist[0])

print('\nTakeaway: SVRG reduces gradient variance enough to make epoch-wise progress nearly geometric on this finite-sum problem.')


---

### Exercise 5: Linear Scaling Rule ★★

Verify the linear scaling rule for learning rate adjustment with batch size.

**(a)** Train a logistic regression model with batch sizes $B \in \{32, 64, 128, 256\}$.

**(b)** Scale the learning rate linearly: $\eta = \eta_0 \cdot B/32$.

**(c)** Verify that the training curves are similar.

In [ ]:
# Exercise 5: Your Solution
np.random.seed(42)
n, d = 10000, 20
A = np.random.randn(n, d)
x_true = np.random.randn(d)
b = A @ x_true + 0.1 * np.random.randn(n)

def f_full(x): return 0.5 * np.mean((A @ x - b)**2)

base_lr = 0.01
base_batch = 32
batch_sizes = [32, 64, 128, 256]
results = {}

for B in batch_sizes:
    lr = base_lr * B / base_batch
    x = np.zeros(d)
    err = []
    for t in range(5000):
        idx = np.random.randint(0, n, size=B)
        g = A[idx].T @ (A[idx] @ x - b[idx]) / B
        x = x - lr * g
        if (t + 1) % 500 == 0:
            err.append(f_full(x))
    results[B] = np.array(err)

print(f"{'Batch':>6} | {'Final loss':>12}")
print('-' * 25)
for B in batch_sizes:
    print(f'{B:>6} | {results[B][-1]:>12.6f}')

In [ ]:
# Exercise 5: Solution
np.random.seed(42)
n, d = 10000, 20
A = np.random.randn(n, d)
x_true = np.random.randn(d)
b = A @ x_true + 0.1 * np.random.randn(n)

def f_full(x): return 0.5 * np.mean((A @ x - b)**2)

base_lr = 0.01
base_batch = 32
batch_sizes = [32, 64, 128, 256]
results = {}

for B in batch_sizes:
    lr = base_lr * B / base_batch
    x = np.zeros(d)
    err = []
    for t in range(5000):
        idx = np.random.randint(0, n, size=B)
        g = A[idx].T @ (A[idx] @ x - b[idx]) / B
        x = x - lr * g
        if (t + 1) % 500 == 0:
            err.append(f_full(x))
    results[B] = np.array(err)

header('Exercise 5: Linear Scaling Rule')
print(f"{'Batch':>6} | {'Final loss':>12}")
print('-' * 25)
for B in batch_sizes:
    print(f'{B:>6} | {results[B][-1]:>12.6f}')

final_losses = [results[B][-1] for B in batch_sizes]
check_true('losses are similar', max(final_losses) / min(final_losses) < 3.0)

if HAS_MPL:
    fig, ax = plt.subplots(figsize=(10, 6))
    steps = np.arange(1, len(results[32]) + 1) * 500
    for B in batch_sizes:
        ax.plot(steps, results[B], label=f'B={B}, lr={0.01*B/32:.4f}')
    ax.set_yscale('log')
    ax.set_title('Linear scaling rule: similar curves with scaled LR')
    ax.set_xlabel('Iteration')
    ax.set_ylabel('Loss')
    ax.legend()
    fig.tight_layout()
    plt.show()

print('\nTakeaway: Scaling LR linearly with batch size maintains similar training dynamics.')

---

### Exercise 6: Momentum in Stochastic Setting ★★

Compare SGD with and without momentum on a noisy quadratic problem.

**(a)** Analyze how momentum affects the variance of the iterates.

**(b)** Compare convergence rates.

**(c)** Find the optimal momentum coefficient.

In [ ]:
# Exercise 6: Your Solution
np.random.seed(42)
d = 20
mu, L, sigma = 0.1, 1.0, 0.5
D = np.linspace(mu, L, d)
x_star = np.random.randn(d)

def grad_sc(x): return D * (x - x_star)
def stochastic_grad(x, sigma):
    return grad_sc(x) + sigma * np.random.randn(d)

T = 3000

# Vanilla SGD
x_sgd = np.zeros(d)
err_sgd = []
for t in range(T):
    x_sgd = x_sgd - 0.01 * stochastic_grad(x_sgd, sigma)
    err_sgd.append(np.linalg.norm(x_sgd - x_star)**2)

# SGD + Momentum with different beta values
betas = [0.0, 0.5, 0.9, 0.99]
err_moms = {}

for beta in betas:
    x = np.zeros(d)
    v = np.zeros(d)
    errs = []
    for t in range(T):
        g = stochastic_grad(x, sigma)
        v = beta * v + g
        x = x - 0.01 * v
        errs.append(np.linalg.norm(x - x_star)**2)
    err_moms[beta] = np.array(errs)

print(f'Final errors:')
print(f'  SGD (beta=0.0): {err_sgd[-1]:.6f}')
for beta in betas:
    print(f'  SGD+Mom (beta={beta}): {err_moms[beta][-1]:.6f}')

In [ ]:
# Exercise 6: Solution
np.random.seed(42)
d = 20
mu, L, sigma = 0.1, 1.0, 0.5
D = np.linspace(mu, L, d)
x_star = np.random.randn(d)


def grad_sc(x):
    return D * (x - x_star)


def stochastic_grad(x, sigma):
    return grad_sc(x) + sigma * np.random.randn(d)


T = 3000
x_sgd = np.zeros(d)
err_sgd = []
for t in range(T):
    x_sgd = x_sgd - 0.01 * stochastic_grad(x_sgd, sigma)
    err_sgd.append(np.linalg.norm(x_sgd - x_star) ** 2)

betas = [0.0, 0.5, 0.9, 0.99]
err_moms = {}
for beta in betas:
    x = np.zeros(d)
    v = np.zeros(d)
    errs = []
    for t in range(T):
        g = stochastic_grad(x, sigma)
        v = beta * v + g
        x = x - 0.01 * v
        errs.append(np.linalg.norm(x - x_star) ** 2)
    err_moms[beta] = np.array(errs)

header('Exercise 6: Momentum in Stochastic Setting')
print('Final errors:')
print(f'  SGD (beta=0.0): {err_sgd[-1]:.6f}')
for beta in betas:
    print(f'  SGD+Mom (beta={beta}): {err_moms[beta][-1]:.6f}')

best_beta = min(betas, key=lambda b: err_moms[b][-1])
print()
print(f'Best beta among tested values: {best_beta}')
check_true('plain SGD is best at this noise level', best_beta == 0.0)
check_true('too much momentum hurts', err_moms[0.99][-1] > err_moms[0.5][-1] > err_moms[0.0][-1])

if HAS_MPL:
    fig, ax = plt.subplots(figsize=(10, 6))
    steps = np.arange(1, T + 1)
    window = 50
    for beta in betas:
        smoothed = np.convolve(err_moms[beta], np.ones(window) / window, mode='valid')
        ax.plot(steps[window - 1:], smoothed, label=f'beta={beta}')
    ax.set_yscale('log')
    ax.set_title('SGD with different momentum values')
    ax.set_xlabel('Iteration')
    ax.set_ylabel('||x - x*||^2')
    ax.legend()
    fig.tight_layout()
    plt.show()
    plt.close('all')

print()
print('Takeaway: In noisy optimization, extra momentum can increase the stationary error unless the step size and noise level are tuned together.')

---

### Exercise 7: Distributed SGD ★★★

Implement distributed SGD with gradient averaging and analyze the speedup.

**(a)** Simulate $K \in \{1, 2, 4, 8\}$ workers with data parallelism.

**(b)** Compare convergence speed (in terms of total gradient evaluations).

**(c)** Analyze the communication-computation trade-off.

In [ ]:
# Exercise 7: Your Solution
np.random.seed(42)
n, d = 10000, 20
A = np.random.randn(n, d)
x_true = np.random.randn(d)
b = A @ x_true + 0.1 * np.random.randn(n)

def f_full(x): return 0.5 * np.mean((A @ x - b)**2)

K_values = [1, 2, 4, 8]
results = {}

for K in K_values:
    data_partitions = np.array_split(np.arange(n), K)
    x_workers = [np.zeros(d) for _ in range(K)]
    lr = 0.001 * K
    
    total_grad_evals = 0
    loss_at_evals = []
    eval_points = [1000, 2000, 5000, 10000, 20000]
    eval_idx = 0
    
    for step in range(25000):
        grads = []
        for k in range(K):
            idx = np.random.choice(data_partitions[k], size=32)
            g = A[idx].T @ (A[idx] @ x_workers[k] - b[idx]) / 32
            grads.append(g)
            total_grad_evals += 32
        
        g_avg = np.mean(grads, axis=0)
        for k in range(K):
            x_workers[k] = x_workers[k] - lr * g_avg
        
        if eval_idx < len(eval_points) and total_grad_evals >= eval_points[eval_idx]:
            loss = f_full(np.mean(x_workers, axis=0))
            loss_at_evals.append((eval_points[eval_idx], loss))
            eval_idx += 1
    
    results[K] = loss_at_evals

print(f'Distributed SGD:')
print(f"{'Grad evals':>12} | {'K=1':>10} | {'K=2':>10} | {'K=4':>10} | {'K=8':>10}")
print('-' * 60)
for i in range(len(results[1])):
    evals = results[1][i][0]
    vals = [results[K][i][1] for K in K_values]
    print(f'{evals:>12} | ' + ' | '.join(f'{v:>10.6f}' for v in vals))

In [ ]:
# Exercise 7: Solution
np.random.seed(42)
n, d = 10000, 20
A = np.random.randn(n, d)
x_true = np.random.randn(d)
b = A @ x_true + 0.1 * np.random.randn(n)

def f_full(x): return 0.5 * np.mean((A @ x - b)**2)

K_values = [1, 2, 4, 8]
results = {}
init_loss = f_full(np.zeros(d))

for K in K_values:
    data_partitions = np.array_split(np.arange(n), K)
    x_workers = [np.zeros(d) for _ in range(K)]
    lr = 0.001 * K

    total_grad_evals = 0
    loss_at_evals = []
    eval_points = [1000, 2000, 5000, 10000, 20000]
    eval_idx = 0

    for step in range(25000):
        grads = []
        for k in range(K):
            idx = np.random.choice(data_partitions[k], size=32)
            g = A[idx].T @ (A[idx] @ x_workers[k] - b[idx]) / 32
            grads.append(g)
            total_grad_evals += 32

        g_avg = np.mean(grads, axis=0)
        for k in range(K):
            x_workers[k] = x_workers[k] - lr * g_avg

        if eval_idx < len(eval_points) and total_grad_evals >= eval_points[eval_idx]:
            loss = f_full(np.mean(x_workers, axis=0))
            loss_at_evals.append((eval_points[eval_idx], loss))
            eval_idx += 1

    results[K] = loss_at_evals

header('Exercise 7: Distributed SGD')
print(f"{'Grad evals':>12} | {'K=1':>10} | {'K=2':>10} | {'K=4':>10} | {'K=8':>10}")
print('-' * 60)
for i in range(len(results[1])):
    evals = results[1][i][0]
    vals = [results[K][i][1] for K in K_values]
    print(f'{evals:>12} | ' + ' | '.join(f'{v:>10.6f}' for v in vals))

final_losses = [results[K][-1][1] for K in K_values]
check_true('all configurations reduce loss substantially', all(l < 0.35 * init_loss for l in final_losses))
check_true('worker counts have similar final losses', max(final_losses) / min(final_losses) < 1.05)

print()
print('Takeaway: Gradient averaging keeps different worker counts on broadly similar optimization trajectories when the learning rate is scaled with K.')

---

### Exercise 8: SGD Implicit Bias ★★★

Train an overparameterized linear model with SGD and full-batch GD. Compare the $\ell_2$ norm of the solutions.

**(a)** Create an underdetermined system ($n < d$).

**(b)** Train with SGD (batch size 1) and full-batch GD from the same initialization.

**(c)** Verify that SGD finds a solution with lower $\ell_2$ norm.

In [ ]:
# Exercise 8: Starter workspace
print('Use this workspace to try your own SGD experiment before running the reference solution below.')


In [ ]:
# Exercise 8: Solution
np.random.seed(42)
n, d = 50, 200
A = np.random.randn(n, d)
x_true = np.random.randn(d)
b = A @ x_true

def f_full(x): return 0.5 * np.mean((A @ x - b)**2)
def grad_full(x): return A.T @ (A @ x - b) / n

x_gd = np.zeros(d)
L = np.linalg.norm(A.T @ A) / n
eta_gd = 0.9 / L
for t in range(10000):
    x_gd = x_gd - eta_gd * grad_full(x_gd)

x_sgd = np.zeros(d)
eta_sgd = 0.005
for t in range(10000 * n):
    i = np.random.randint(n)
    g = (A[i] @ x_sgd - b[i]) * A[i]
    x_sgd = x_sgd - eta_sgd * g

header('Exercise 8: SGD Implicit Bias')
print(f'Full-batch GD:')
print(f'  ||x_gd|| = {np.linalg.norm(x_gd):.6f}')
print(f'  ||Ax - b|| = {np.linalg.norm(A @ x_gd - b):.6e}')

print()
print('SGD:')
print(f'  ||x_sgd|| = {np.linalg.norm(x_sgd):.6f}')
print(f'  ||Ax - b|| = {np.linalg.norm(A @ x_sgd - b):.6e}')

norm_ratio = np.linalg.norm(x_sgd) / np.linalg.norm(x_gd)
print()
print(f'Norm ratio: ||x_sgd|| / ||x_gd|| = {norm_ratio:.4f}')

check_true('GD achieves near-zero residual', np.linalg.norm(A @ x_gd - b) < 1e-6)
check_true('SGD achieves near-zero residual', np.linalg.norm(A @ x_sgd - b) < 1e-6)
check_true('SGD and GD have nearly identical norms', abs(norm_ratio - 1.0) < 0.05)

print()
print('Note: In this overparameterized linear system, both methods converge to essentially the same minimum-norm interpolating solution from zero initialization.')
print(f'SGD norm: {np.linalg.norm(x_sgd):.4f}, GD norm: {np.linalg.norm(x_gd):.4f}')

print()
print('Takeaway: The implicit bias story depends on the model and initialization; here, SGD and full GD share the same minimum-norm bias.')

---

## What to Review After Finishing

- [ ] Exercise 1: Mini-batch gradient is unbiased with variance σ²/B
- [ ] Exercise 2: SGD converges at rate O(1/t) for strongly convex problems
- [ ] Exercise 3: Variance decreases linearly with batch size
- [ ] Exercise 4: SVRG achieves linear convergence for finite-sum problems
- [ ] Exercise 5: Linear scaling rule maintains similar training dynamics
- [ ] Exercise 6: Momentum reduces variance but too much causes oscillations
- [ ] Exercise 7: Distributed SGD with gradient averaging converges across workers
- [ ] Exercise 8: SGD implicit bias toward simpler solutions

## References

1. Bottou, L. (2010). "Large-scale machine learning with stochastic gradient descent." COMPSTAT.
2. Johnson, R. & Zhang, T. (2013). "Accelerating stochastic gradient descent using predictive variance reduction." NeurIPS.
3. Defazio, A., Bach, F. & Lacoste-Julien, S. (2014). "SAGA: A fast incremental gradient method." NeurIPS.